In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load and clean the data
data = pd.read_csv('s10_ex08.csv')
data.columns = data.columns.str.strip()

# Rename index column for clarity
data.rename(columns={'Unnamed: 0': 'Sample Index'}, inplace=True)

# Choose a channel to analyze (e.g., 'P4')
channel = 'P4'
eeg_signal = data[channel].values

# Sampling frequency (assumed; adjust if known)
fs = 256  
N = len(eeg_signal)

# Apply DFT
dft = np.fft.fft(eeg_signal)
frequencies = np.fft.fftfreq(N, d=1/fs)

# Filter to positive frequencies only
positive_mask = frequencies >= 0
frequencies = frequencies[positive_mask]
dft_magnitude = np.abs(dft)[positive_mask]

# Define Alpha and Beta bands
alpha_band = (frequencies >= 8) & (frequencies <= 13)
beta_band = (frequencies > 13) & (frequencies <= 30)
prominent_band = alpha_band | beta_band

# Filter the DFT
filtered_dft = np.zeros_like(dft, dtype=complex)
filtered_dft[positive_mask] = dft[positive_mask] * prominent_band
filtered_dft[~positive_mask] = np.conj(filtered_dft[positive_mask][1:][::-1])

# Reconstruct signal using inverse FFT
reconstructed_signal = np.fft.ifft(filtered_dft).real
time_axis = np.arange(N) / fs

# Plot original signal
plt.figure(figsize=(12, 4))
plt.plot(time_axis, eeg_signal)
plt.title(f"Original EEG Signal ({channel})")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (uV)")
plt.grid(True)
plt.show()

# Plot reconstructed signal
plt.figure(figsize=(12, 4))
plt.plot(time_axis, reconstructed_signal)
plt.title(f"Reconstructed EEG ({channel} - Alpha + Beta Waves)")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (uV)")
plt.grid(True)
plt.show()

ValueError: NumPy boolean array indexing assignment cannot assign 11999 input values to the 12000 output values where the mask is true